## 1. Verificación inicial de los archivos del proyecto

### Objetivo

Comprobar la disponibilidad de los principales archivos generados durante el estudio, como punto de partida para una revisión independiente de la consistencia y reproducibilidad del análisis.

### Procedimiento

Se identifica la carpeta raíz del proyecto y se verifica la existencia de seis archivos: el conjunto de datos preparado para el modelado, el inventario de variables predictoras, las predicciones fuera de muestra, la configuración del modelo seleccionado, el modelo entrenado y las métricas obtenidas sobre el conjunto de prueba.

También se contabilizan los checkpoints correspondientes a la evaluación temporal.

### Alcance de la comprobación

Esta verificación permite confirmar la disponibilidad de los archivos, pero no acredita todavía su integridad, la correspondencia entre sus contenidos ni la validez de los resultados.

El procedimiento es exclusivamente de lectura. No modifica los datos, no reentrena modelos y no altera los resultados originales.


In [ ]:

from pathlib import Path

# Localizar la raíz del proyecto
carpeta_actual = Path.cwd().resolve()

candidatas = [
    carpeta_actual,
    carpeta_actual.parent
]

RUTA_PROYECTO = next(
    (
        ruta for ruta in candidatas
        if (ruta / "notebooks").is_dir()
        and (ruta / "resultados").is_dir()
    ),
    None
)

if RUTA_PROYECTO is None:
    raise FileNotFoundError(
        "No se encuentra la raíz del Proyecto_Churn_."
    )

# Archivos que se comprobarán
archivos_verificacion = {
    "Dataset de modelado de 90 días":
        RUTA_PROYECTO / "resultados" / "dataset_modelado_90d.pkl",

    "Inventario de las 176 variables":
        RUTA_PROYECTO / "resultados" / "columnas_predictoras_176_90d.csv",

    "Predicciones temporales OOF":
        RUTA_PROYECTO / "resultados" / "modelado_predicciones_oof_temporales.pkl",

    "Configuración final del modelo":
        RUTA_PROYECTO / "resultados" / "modelado_configuracion_final_60d.json",

    "Modelo final entrenado":
        RUTA_PROYECTO / "modelos" / "modelo_final_logistic_l2_60d.joblib",

    "Métricas del test externo":
        RUTA_PROYECTO / "resultados" / "modelado_metricas_test_externo_60d.csv"
}

print("VERIFICACIÓN INICIAL DE ARCHIVOS")
print("-" * 55)

for nombre, ruta in archivos_verificacion.items():
    estado = "OK" if ruta.is_file() else "NO ENCONTRADO"
    print(f"{nombre}: {estado}")

# Comprobar los checkpoints temporales
carpeta_checkpoints = (
    RUTA_PROYECTO
    / "resultados"
    / "checkpoints_oof_temporal"
)

numero_checkpoints = (
    len(list(carpeta_checkpoints.glob("oof_*d_*.pkl")))
    if carpeta_checkpoints.is_dir()
    else 0
)

print("-" * 55)
print(f"Checkpoints temporales encontrados: {numero_checkpoints}")

print("\nComprobación de existencia finalizada.")
print("No se ha modificado ningún archivo.")


### Interpretación de la comprobación 1

Se ha confirmado la existencia de los seis archivos principales examinados y de nueve checkpoints temporales.

Este resultado acredita su disponibilidad, pero no permite todavía establecer la integridad de sus contenidos ni la correspondencia con las predicciones consolidadas.

## 2. Verificación de los checkpoints temporales

### Objetivo

Comprobar la consistencia de los nueve archivos de predicciones fuera de muestra (*out-of-fold*, OOF), correspondientes a las ventanas de 30, 60 y 90 días y a los tres modelos finalistas: regresión logística L2, regresión logística con ponderación de clases y XGBoost.

### Procedimiento

Para cada checkpoint se comprueba su estructura, el número y orden de clientes, la correspondencia de las etiquetas reales de churn y la validez de las probabilidades estimadas.

También se contrastan las probabilidades almacenadas con las del archivo consolidado de predicciones OOF temporales.

### Alcance

La comprobación permite detectar discrepancias entre los checkpoints y los resultados consolidados, pero no demuestra por sí sola que las predicciones se hayan generado con los hiperparámetros, los datos o las particiones de validación declarados.

El procedimiento es exclusivamente de lectura y no modifica los archivos originales.


In [ ]:

import numpy as np
import pandas as pd

# Archivo consolidado de predicciones temporales
ruta_predicciones = (
    RUTA_PROYECTO
    / "resultados"
    / "modelado_predicciones_oof_temporales.pkl"
)

predicciones = pd.read_pickle(ruta_predicciones)

ventanas = [30, 60, 90]
modelos = [
    "logistic_l2",
    "logistic_balanced",
    "xgboost"
]

columnas_checkpoint = [
    "customer_id",
    "churn_real",
    "probabilidad"
]

# Validación de la tabla de referencia
if predicciones.shape != (44_203, 11):
    raise ValueError(
        "Dimensiones inesperadas en las predicciones OOF."
    )

if (
    predicciones["customer_id"].isna().any()
    or predicciones["customer_id"].duplicated().any()
):
    raise ValueError(
        "Identificadores nulos o duplicados en la referencia."
    )

if not predicciones["churn_real"].isin([0, 1]).all():
    raise ValueError(
        "Etiquetas churn no binarias en la referencia."
    )

resultados = []

print("VERIFICACIÓN DE LOS NUEVE CHECKPOINTS")
print("-" * 65)

for ventana in ventanas:
    for modelo in modelos:

        nombre = f"oof_{ventana}d_{modelo}.pkl"
        ruta = carpeta_checkpoints / nombre
        columna_oof = (
            f"probabilidad__{ventana}d__{modelo}"
        )

        comprobaciones = {}

        if not ruta.is_file():
            comprobaciones["archivo"] = False
        else:
            checkpoint = pd.read_pickle(ruta)

            comprobaciones["estructura"] = (
                isinstance(checkpoint, pd.DataFrame)
                and list(checkpoint.columns)
                == columnas_checkpoint
                and checkpoint.shape == (44_203, 3)
            )

            if comprobaciones["estructura"]:

                comprobaciones["clientes"] = (
                    np.array_equal(
                        checkpoint["customer_id"].to_numpy(),
                        predicciones["customer_id"].to_numpy()
                    )
                )

                comprobaciones["etiquetas"] = (
                    np.array_equal(
                        checkpoint["churn_real"].to_numpy(),
                        predicciones["churn_real"].to_numpy()
                    )
                )

                probabilidades = pd.to_numeric(
                    checkpoint["probabilidad"],
                    errors="coerce"
                ).to_numpy(dtype=float)

                comprobaciones["probabilidades_validas"] = (
                    np.isfinite(probabilidades).all()
                    and ((probabilidades >= 0)
                         & (probabilidades <= 1)).all()
                )

                comprobaciones["coincidencia_oof"] = (
                    columna_oof in predicciones.columns
                    and np.array_equal(
                        probabilidades,
                        pd.to_numeric(
                            predicciones[columna_oof],
                            errors="coerce"
                        ).to_numpy(dtype=float)
                    )
                )

        correcto = all(comprobaciones.values())

        resultados.append({
            "checkpoint": nombre,
            "correcto": correcto,
            "comprobaciones": comprobaciones
        })

        estado = "OK" if correcto else "REVISAR"
        print(f"{nombre}: {estado}")

        if not correcto:
            for prueba, superada in comprobaciones.items():
                if not superada:
                    print(f"  Incidencia: {prueba}")

print("-" * 65)

total_correctos = sum(
    resultado["correcto"]
    for resultado in resultados
)

print(f"Checkpoints correctos: {total_correctos}/9")

if total_correctos == 9:
    print("Consistencia interna: SUPERADA")
else:
    print("Consistencia interna: REQUIERE REVISIÓN")

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 2

Los nueve checkpoints temporales han superado las comprobaciones de estructura, correspondencia de clientes, etiquetas de churn y validez de las probabilidades.

Asimismo, las probabilidades almacenadas coinciden con las incluidas en el archivo consolidado de predicciones OOF.

Estos resultados confirman la consistencia interna de los archivos examinados. No acreditan todavía que las predicciones se hayan generado con la configuración metodológica declarada ni descartan posibles fugas de información.

## 3. Reproducción independiente de las métricas temporales

### Objetivo

Verificar que las métricas de los nueve modelos evaluados en las ventanas de 30, 60 y 90 días se corresponden con las predicciones OOF almacenadas.

### Procedimiento

Se recalculan las métricas de discriminación, calibración y clasificación a partir de las probabilidades guardadas y las etiquetas reales.

Para cada configuración se utiliza el umbral de decisión registrado en la comparación original. Los valores calculados se contrastan con las métricas conservadas en el archivo de resultados.

### Alcance

Esta comprobación permite verificar la coherencia numérica de los resultados, pero no constituye una nueva validación cruzada ni demuestra por sí sola la ausencia de fugas de información.

No se entrenan modelos ni se modifican los archivos originales.


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    balanced_accuracy_score,
    confusion_matrix
)

# Cargar las evidencias originales
ruta_comparacion = (
    RUTA_PROYECTO
    / "resultados"
    / "modelado_comparacion_temporal_oof.csv"
)

ruta_oof = (
    RUTA_PROYECTO
    / "resultados"
    / "modelado_predicciones_oof_temporales.pkl"
)

comparacion = pd.read_csv(ruta_comparacion)
oof = pd.read_pickle(ruta_oof)

if (
    len(comparacion) != 9
    or comparacion[
        ["ventana_dias", "modelo"]
    ].duplicated().any()
):
    raise ValueError(
        "La tabla de comparación no contiene "
        "nueve configuraciones únicas."
    )

y_real = oof["churn_real"].to_numpy(dtype=int)

metricas_decimales = [
    "average_precision",
    "auc_roc",
    "brier",
    "log_loss",
    "precision",
    "recall",
    "especificidad",
    "f1",
    "f2",
    "balanced_accuracy",
    "tasa_alertas"
]

metricas_enteras = [
    "verdaderos_negativos",
    "falsos_positivos",
    "falsos_negativos",
    "verdaderos_positivos"
]

resultados_verificacion = []

print("VERIFICACIÓN DE MÉTRICAS TEMPORALES")
print("-" * 65)

for _, fila in comparacion.iterrows():

    ventana = int(fila["ventana_dias"])
    modelo = str(fila["modelo"])
    umbral = float(fila["umbral_f2"])

    columna = f"probabilidad__{ventana}d__{modelo}"

    probabilidades = (
        oof[columna].to_numpy(dtype=float)
    )

    predichas = (
        probabilidades >= umbral
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_real,
        predichas,
        labels=[0, 1]
    ).ravel()

    calculadas = {
        "average_precision": average_precision_score(
            y_real, probabilidades
        ),
        "auc_roc": roc_auc_score(
            y_real, probabilidades
        ),
        "brier": float(
            np.mean((probabilidades - y_real) ** 2)
        ),
        "log_loss": log_loss(
            y_real, probabilidades, labels=[0, 1]
        ),
        "precision": precision_score(
            y_real, predichas, zero_division=0
        ),
        "recall": recall_score(
            y_real, predichas, zero_division=0
        ),
        "especificidad": float(tn / (tn + fp)),
        "f1": f1_score(
            y_real, predichas, zero_division=0
        ),
        "f2": fbeta_score(
            y_real, predichas, beta=2, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_real, predichas
        ),
        "tasa_alertas": float(predichas.mean()),
        "verdaderos_negativos": int(tn),
        "falsos_positivos": int(fp),
        "falsos_negativos": int(fn),
        "verdaderos_positivos": int(tp)
    }

    discrepancias = []

    for metrica in metricas_decimales:
        if not np.isclose(
            calculadas[metrica],
            float(fila[metrica]),
            rtol=1e-8,
            atol=1e-8
        ):
            discrepancias.append(metrica)

    for metrica in metricas_enteras:
        if (
            calculadas[metrica]
            != int(fila[metrica])
        ):
            discrepancias.append(metrica)

    correcto = len(discrepancias) == 0

    resultados_verificacion.append(correcto)

    estado = "OK" if correcto else "REVISAR"
    print(f"{ventana}d | {modelo}: {estado}")

    if discrepancias:
        print(
            "  Métricas discrepantes:",
            ", ".join(discrepancias)
        )

print("-" * 65)

total = sum(resultados_verificacion)

print(f"Configuraciones verificadas: {total}/9")

if total == 9:
    print("CONSISTENCIA NUMÉRICA: SUPERADA")
else:
    print("CONSISTENCIA NUMÉRICA: REQUIERE REVISIÓN")

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 3

Las métricas de las nueve configuraciones evaluadas coinciden con las recalculadas a partir de las predicciones OOF y sus umbrales de decisión.

Esta comprobación confirma la consistencia numérica de los resultados almacenados, pero no verifica todavía el procedimiento de generación de las predicciones.

## 4. Verificación independiente de las particiones de datos

### Objetivo

Comprobar que la separación entre entrenamiento, desarrollo, holdout interno y prueba externa es consistente con el procedimiento descrito en el análisis original.

### Procedimiento

Se reconstruye la partición interna mediante StratifiedShuffleSplit, utilizando una proporción del 15 % y una semilla aleatoria de 314159.

Los resultados se contrastan con el manifiesto de particiones almacenado. También se verifica la ausencia de identificadores compartidos entre entrenamiento y prueba externa, así como la correspondencia de los clientes del conjunto de desarrollo con las predicciones OOF consolidadas.

### Alcance

La comprobación permite verificar la reproducibilidad de las particiones y su consistencia con las evidencias conservadas.

No constituye una repetición del entrenamiento ni acredita, por sí sola, la ausencia de fugas de información durante la construcción de las variables predictoras.


In [ ]:

from sklearn.model_selection import StratifiedShuffleSplit

# Archivos originales
ruta_dataset = (
    RUTA_PROYECTO / "resultados" / "dataset_modelado_90d.pkl"
)

ruta_manifiesto = (
    RUTA_PROYECTO / "resultados" / "split_manifest_v1.csv"
)

ruta_oof = (
    RUTA_PROYECTO
    / "resultados"
    / "modelado_predicciones_oof_temporales.pkl"
)

# Comprobar disponibilidad
for ruta in [ruta_dataset, ruta_manifiesto, ruta_oof]:
    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encuentra: {ruta.name}"
        )

# Leer evidencias sin modificarlas
datos = pd.read_pickle(ruta_dataset)
manifiesto = pd.read_csv(
    ruta_manifiesto,
    dtype={"customer_id": "string"}
)
oof = pd.read_pickle(ruta_oof)

# Recuperar las particiones originales
train = (
    datos.loc[
        datos["particion"] == "train",
        ["customer_id", "churn"]
    ]
    .reset_index(drop=True)
)

test = (
    datos.loc[
        datos["particion"] == "test",
        ["customer_id", "churn"]
    ]
    .reset_index(drop=True)
)

# Reconstruir el holdout interno
divisor = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=314159
)

indices_desarrollo, indices_holdout = next(
    divisor.split(
        train[["customer_id"]],
        train["churn"]
    )
)

# Reconstruir las etiquetas de partición
split_esperado = pd.Series(
    "development",
    index=train.index
)

split_esperado.iloc[
    indices_holdout
] = "locked_internal_holdout"

# Normalizar identificadores para la comparación
ids_train = train["customer_id"].astype("string")
ids_test = test["customer_id"].astype("string")
ids_manifiesto = manifiesto["customer_id"].astype("string")
ids_oof = oof["customer_id"].astype("string")

ids_desarrollo = ids_train.iloc[
    indices_desarrollo
].reset_index(drop=True)

etiquetas_desarrollo = train["churn"].iloc[
    indices_desarrollo
].reset_index(drop=True)

# Comprobaciones independientes
pruebas = {
    "Total de 65.005 clientes": (
        len(datos) == 65_005
    ),
    "Entrenamiento de 52.004 clientes": (
        len(train) == 52_004
    ),
    "Test externo de 13.001 clientes": (
        len(test) == 13_001
    ),
    "Identificadores únicos": (
        datos["customer_id"].notna().all()
        and not datos["customer_id"].duplicated().any()
    ),
    "Sin solapamiento train/test": (
        set(ids_train).isdisjoint(set(ids_test))
    ),
    "Manifiesto de 52.004 registros": (
        len(manifiesto) == 52_004
    ),
    "Orden de clientes del manifiesto": (
        ids_manifiesto.reset_index(drop=True).equals(
            ids_train.reset_index(drop=True)
        )
    ),
    "Etiquetas churn del manifiesto": (
        np.array_equal(
            manifiesto["churn"].to_numpy(),
            train["churn"].to_numpy()
        )
    ),
    "Reproducción del split interno": (
        manifiesto["split"].reset_index(drop=True).equals(
            split_esperado.reset_index(drop=True)
        )
    ),
    "Tamaño del desarrollo": (
        len(indices_desarrollo) == 44_203
    ),
    "Tamaño del holdout interno": (
        len(indices_holdout) == 7_801
    ),
    "Clientes OOF alineados con desarrollo": (
        ids_oof.reset_index(drop=True).equals(
            ids_desarrollo
        )
    ),
    "Etiquetas OOF alineadas con desarrollo": (
        np.array_equal(
            oof["churn_real"].to_numpy(),
            etiquetas_desarrollo.to_numpy()
        )
    )
}

print("VERIFICACIÓN DE PARTICIONES")
print("-" * 65)

for nombre, correcto in pruebas.items():
    estado = "OK" if correcto else "REVISAR"
    print(f"{nombre}: {estado}")

print("-" * 65)

total_correctas = sum(pruebas.values())

print(
    f"Comprobaciones superadas: "
    f"{total_correctas}/{len(pruebas)}"
)

if all(pruebas.values()):
    print("CONSISTENCIA DE PARTICIONES: SUPERADA")
else:
    print("CONSISTENCIA DE PARTICIONES: REQUIERE REVISIÓN")

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 4

Se han superado las trece comprobaciones sobre los tamaños de las particiones, la unicidad de los identificadores, la ausencia de solapamiento entre entrenamiento y prueba externa y la reproducción del holdout interno.

También se ha confirmado la correspondencia entre los clientes del conjunto de desarrollo y las predicciones OOF almacenadas.

## 5. Verificación de los cinco pliegues de validación cruzada

### Objetivo

Reconstruir la división estratificada del conjunto de desarrollo en cinco pliegues y comprobar su consistencia estructural.

### Procedimiento

Se utiliza StratifiedKFold con cinco pliegues, mezcla aleatoria y semilla 314159, conforme a la configuración documentada en el notebook original de modelado.

Se comprueba que cada cliente aparezca exactamente una vez en validación, que los subconjuntos de entrenamiento y validación de cada pliegue no se solapen y que se mantenga una distribución similar de la clase churn.

También se verifica que ningún cliente de validación pertenezca al holdout interno ni al conjunto de prueba externa.

### Alcance

La reconstrucción permite verificar la coherencia del esquema de validación definido, pero no demuestra que los checkpoints se hayan generado utilizando exactamente esos pliegues. Para acreditar este extremo sería necesario disponer de evidencias adicionales de ejecución o reproducir el entrenamiento.

La comprobación no modifica datos, modelos ni resultados.


In [ ]:

from sklearn.model_selection import StratifiedKFold

# Etiquetas e identificadores del desarrollo
y_desarrollo_cv = oof["churn_real"].to_numpy(dtype=int)
ids_desarrollo_cv = oof["customer_id"].astype(str).to_numpy()

# Clientes reservados fuera del desarrollo
ids_reservados = set(
    ids_train.iloc[indices_holdout].astype(str)
).union(
    set(ids_test.astype(str))
)

# Configuración original
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=314159
)

# Registro de apariciones en validación
apariciones_validacion = np.zeros(
    len(y_desarrollo_cv),
    dtype=int
)

resumen = []
sin_solapamientos = True
sin_clientes_reservados = True
pliegues_con_ambas_clases = True

for numero, (idx_train, idx_val) in enumerate(
    cv.split(
        np.zeros(len(y_desarrollo_cv)),
        y_desarrollo_cv
    ),
    start=1
):
    apariciones_validacion[idx_val] += 1

    # Comprobar separación interna
    sin_solapamientos &= (
        len(np.intersect1d(idx_train, idx_val)) == 0
        and len(idx_train) + len(idx_val)
        == len(y_desarrollo_cv)
    )

    # Comprobar exclusión de clientes reservados
    sin_clientes_reservados &= set(
        ids_desarrollo_cv[idx_val]
    ).isdisjoint(ids_reservados)

    # Comprobar presencia de las dos clases
    pliegues_con_ambas_clases &= (
        len(np.unique(y_desarrollo_cv[idx_val])) == 2
    )

    resumen.append({
        "pliegue": numero,
        "entrenamiento": len(idx_train),
        "validacion": len(idx_val),
        "tasa_churn": y_desarrollo_cv[idx_val].mean()
    })

resumen = pd.DataFrame(resumen)

diferencia_churn = (
    resumen["tasa_churn"].max()
    - resumen["tasa_churn"].min()
)

pruebas_cv = {
    "Cinco pliegues generados": len(resumen) == 5,
    "Cada cliente validado exactamente una vez": (
        np.all(apariciones_validacion == 1)
    ),
    "Sin solapamiento entrenamiento/validación": (
        sin_solapamientos
    ),
    "Sin clientes del holdout o test externo": (
        sin_clientes_reservados
    ),
    "Ambas clases presentes en cada pliegue": (
        pliegues_con_ambas_clases
    ),
    "Diferencia de prevalencia inferior a 0,1 pp": (
        diferencia_churn < 0.001
    )
}

print("VERIFICACIÓN DE VALIDACIÓN CRUZADA")
print("-" * 65)

for _, fila in resumen.iterrows():
    print(
        f"Pliegue {int(fila['pliegue'])}: "
        f"train={int(fila['entrenamiento'])}, "
        f"validación={int(fila['validacion'])}, "
        f"churn={fila['tasa_churn']:.4%}"
    )

print("-" * 65)

for nombre, correcto in pruebas_cv.items():
    print(f"{nombre}: {'OK' if correcto else 'REVISAR'}")

print("-" * 65)

total_cv = sum(pruebas_cv.values())

print(
    f"Comprobaciones superadas: "
    f"{total_cv}/{len(pruebas_cv)}"
)

if all(pruebas_cv.values()):
    print("ESTRUCTURA DE VALIDACIÓN CRUZADA: SUPERADA")
else:
    print("ESTRUCTURA DE VALIDACIÓN CRUZADA: REQUIERE REVISIÓN")

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 5

Se han superado las seis comprobaciones relativas a la reconstrucción de los cinco pliegues de validación cruzada estratificada.

Cada cliente aparece exactamente una vez en validación, no existen solapamientos entre los subconjuntos y la prevalencia de churn es similar en todos los pliegues.

Esta evidencia confirma la coherencia estructural del esquema reconstruido, aunque no acredita por sí sola que los checkpoints se generaran utilizando exactamente esas particiones.

## 6. Verificación del modelo final y sus predicciones

### Objetivo

Comprobar que el modelo final almacenado reproduce las predicciones y las principales métricas registradas para el conjunto de prueba externa.

### Procedimiento

Se recupera el modelo entrenado y se verifica su configuración: regresión logística L2, ventana de 60 días, 176 variables predictoras y umbral de decisión de 0,20.

A continuación, se utiliza el modelo guardado para obtener nuevamente las probabilidades de los 13.001 clientes del conjunto de prueba. Estas se comparan con las predicciones originales y se recalculan las principales métricas de evaluación.

### Alcance

La comprobación permite verificar la correspondencia entre el modelo guardado, los datos de prueba y los resultados conservados.

No supone un nuevo entrenamiento, una nueva selección de hiperparámetros ni una modificación del modelo. Tampoco demuestra por sí sola que todas las decisiones anteriores al uso del test externo se adoptaran sin consultar sus resultados.


In [ ]:

import joblib

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    fbeta_score
)

# Rutas de las evidencias originales
ruta_modelo_final = (
    RUTA_PROYECTO
    / "modelos"
    / "modelo_final_logistic_l2_60d.joblib"
)

ruta_datos_60d = (
    RUTA_PROYECTO
    / "resultados"
    / "dataset_modelado_60d.pkl"
)

ruta_predicciones_test = (
    RUTA_PROYECTO
    / "resultados"
    / "modelado_predicciones_test_externo_60d.csv"
)

ruta_metricas_test = (
    RUTA_PROYECTO
    / "resultados"
    / "modelado_metricas_test_externo_60d.csv"
)

# Cargar modelo y evidencias
paquete = joblib.load(ruta_modelo_final)
modelo_guardado = paquete["modelo"]

datos_60d = pd.read_pickle(ruta_datos_60d)
predicciones_guardadas = pd.read_csv(
    ruta_predicciones_test,
    dtype={"customer_id": "string"}
)

metricas_guardadas = pd.read_csv(
    ruta_metricas_test
).iloc[0]

inventario = pd.read_csv(
    RUTA_PROYECTO
    / "resultados"
    / "columnas_predictoras_176_90d.csv"
)

variables_esperadas = inventario["variable"].tolist()
variables_modelo = list(paquete["variables"])

# Recuperar exclusivamente el test externo
datos_test_60d = datos_60d.loc[
    datos_60d["particion"] == "test"
].copy()

X_test_verificacion = datos_test_60d[
    variables_modelo
]

y_test_verificacion = datos_test_60d[
    "churn"
].to_numpy(dtype=int)

# Reproducir predicciones sin reentrenamiento
probabilidades_recalculadas = (
    modelo_guardado.predict_proba(
        X_test_verificacion
    )[:, 1]
)

umbral_modelo = float(paquete["umbral"])

clases_recalculadas = (
    probabilidades_recalculadas >= umbral_modelo
).astype(int)

# Métricas independientes
ap_recalculada = average_precision_score(
    y_test_verificacion,
    probabilidades_recalculadas
)

auc_recalculada = roc_auc_score(
    y_test_verificacion,
    probabilidades_recalculadas
)

f2_recalculada = fbeta_score(
    y_test_verificacion,
    clases_recalculadas,
    beta=2,
    zero_division=0
)

brier_recalculado = float(
    np.mean(
        (
            probabilidades_recalculadas
            - y_test_verificacion
        ) ** 2
    )
)

# Comparaciones
pruebas_modelo = {
    "Modelo logistic_l2": (
        paquete["modelo_nombre"] == "logistic_l2"
    ),
    "Ventana de 60 días": (
        paquete["ventana_dias"] == 60
    ),
    "Umbral de decisión 0,20": (
        np.isclose(umbral_modelo, 0.20)
    ),
    "176 variables predictoras": (
        len(variables_modelo) == 176
    ),
    "Inventario de variables coincidente": (
        variables_modelo == variables_esperadas
    ),
    "Entrenamiento declarado de 52.004 clientes": (
        paquete["entrenamiento_clientes"] == 52_004
    ),
    "Test externo de 13.001 clientes": (
        len(datos_test_60d) == 13_001
        and len(predicciones_guardadas) == 13_001
    ),
    "Identificadores alineados": (
        np.array_equal(
            datos_test_60d[
                "customer_id"
            ].astype("string").to_numpy(),
            predicciones_guardadas[
                "customer_id"
            ].to_numpy()
        )
    ),
    "Etiquetas reales coincidentes": (
        np.array_equal(
            y_test_verificacion,
            predicciones_guardadas[
                "churn_real"
            ].to_numpy(dtype=int)
        )
    ),
    "Probabilidades reproducidas": (
        np.allclose(
            probabilidades_recalculadas,
            predicciones_guardadas[
                "probabilidad_churn"
            ].to_numpy(dtype=float),
            rtol=1e-10,
            atol=1e-12
        )
    ),
    "Clasificaciones reproducidas": (
        np.array_equal(
            clases_recalculadas,
            predicciones_guardadas[
                "prediccion_churn"
            ].to_numpy(dtype=int)
        )
    ),
    "Average precision coincidente": (
        np.isclose(
            ap_recalculada,
            metricas_guardadas["average_precision"],
            atol=1e-8
        )
    ),
    "AUC-ROC coincidente": (
        np.isclose(
            auc_recalculada,
            metricas_guardadas["auc_roc"],
            atol=1e-8
        )
    ),
    "F2 coincidente": (
        np.isclose(
            f2_recalculada,
            metricas_guardadas["f2"],
            atol=1e-8
        )
    ),
    "Brier score coincidente": (
        np.isclose(
            brier_recalculado,
            metricas_guardadas["brier"],
            atol=1e-8
        )
    )
}

print("VERIFICACIÓN DEL MODELO FINAL")
print("-" * 65)

for nombre, correcto in pruebas_modelo.items():
    print(
        f"{nombre}: "
        f"{'OK' if correcto else 'REVISAR'}"
    )

print("-" * 65)

total_correctas = sum(pruebas_modelo.values())

print(
    f"Comprobaciones superadas: "
    f"{total_correctas}/{len(pruebas_modelo)}"
)

if all(pruebas_modelo.values()):
    print("REPRODUCCIÓN DEL MODELO FINAL: SUPERADA")
else:
    print("REPRODUCCIÓN DEL MODELO FINAL: REQUIERE REVISIÓN")

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 6

Se han superado las quince comprobaciones del modelo final.

El modelo almacenado reproduce las probabilidades, las clasificaciones y las métricas contrastadas sobre los 13.001 clientes del conjunto de prueba externa.

Esto confirma la correspondencia numérica entre el modelo guardado y los resultados conservados, sin necesidad de volver a entrenarlo.

## 7. Verificación de la regla de selección del modelo definitivo

### Objetivo

Comprobar que la selección del modelo definitivo se corresponde con los criterios documentados para comparar las nueve configuraciones temporales.

### Procedimiento

Se recuperan el protocolo metodológico, la tabla de comparación temporal y los documentos de selección del modelo.

Se identifica la configuración de referencia por average precision y se vuelven a aplicar las tolerancias de equivalencia registradas. Entre las configuraciones equivalentes se prioriza la ventana de observación más corta y, posteriormente, el modelo más sencillo según el orden establecido.

El resultado se contrasta con la configuración definitiva conservada.

### Alcance

Esta comprobación permite verificar que la decisión final es coherente con la regla documentada y los resultados almacenados.

No constituye una prueba independiente de que el protocolo se estableciera antes de conocer los resultados ni acredita, por sí sola, que no se consultara el conjunto de prueba durante las decisiones anteriores.


In [ ]:

import json

# Recuperar las evidencias de selección
carpeta_resultados = RUTA_PROYECTO / "resultados"

with open(
    carpeta_resultados / "modelado_protocolo_comparacion_temporal.json",
    encoding="utf-8"
) as archivo:
    protocolo = json.load(archivo)

with open(
    carpeta_resultados / "modelado_propuesta_configuracion_temporal.json",
    encoding="utf-8"
) as archivo:
    propuesta = json.load(archivo)

with open(
    carpeta_resultados / "modelado_configuracion_final_60d.json",
    encoding="utf-8"
) as archivo:
    configuracion_final = json.load(archivo)

tabla = pd.read_csv(
    carpeta_resultados / "modelado_comparacion_temporal_oof.csv"
)

# Recuperar los criterios documentados
tolerancias = protocolo["equivalence_tolerances"]
preferencias = protocolo["model_preference_if_equivalent"]

metricas_equivalencia = [
    "average_precision",
    "auc_roc",
    "f2",
    "recall"
]

# Identificar la configuración de referencia
referencia = (
    tabla.sort_values(
        ["average_precision", "auc_roc"],
        ascending=False
    )
    .iloc[0]
)

# Recalcular las equivalencias
equivalente = pd.Series(True, index=tabla.index)

for metrica in metricas_equivalencia:
    diferencia = referencia[metrica] - tabla[metrica]
    equivalente &= diferencia <= tolerancias[metrica]

# Seleccionar la configuración equivalente preferida
equivalentes = tabla.loc[equivalente].copy()

orden_modelos = {
    nombre: posicion
    for posicion, nombre in enumerate(preferencias)
}

equivalentes["preferencia_modelo"] = (
    equivalentes["modelo"].map(orden_modelos)
)

seleccion = (
    equivalentes.sort_values(
        [
            "ventana_dias",
            "preferencia_modelo",
            "average_precision"
        ],
        ascending=[True, True, False]
    )
    .iloc[0]
)

# Contrastar las evidencias
propuesta_guardada = propuesta["proposed_configuration"]

pruebas_seleccion = {
    "Nueve configuraciones únicas": (
        len(tabla) == 9
        and not tabla[
            ["ventana_dias", "modelo"]
        ].duplicated().any()
    ),
    "Cinco pliegues registrados": (
        protocolo["cv_folds"] == 5
    ),
    "Semilla 314159 registrada": (
        protocolo["cv_random_state"] == 314159
    ),
    "Selección sobre desarrollo declarada": (
        protocolo["selection_data"] == "development_only"
    ),
    "Referencia 90 días XGBoost": (
        int(referencia["ventana_dias"]) == 90
        and referencia["modelo"] == "xgboost"
    ),
    "Equivalencias almacenadas coincidentes": (
        np.array_equal(
            equivalente.to_numpy(),
            tabla["equivalente_a_referencia"]
            .astype(str)
            .str.lower()
            .eq("true")
            .to_numpy()
        )
    ),
    "Propuesta temporal reproducida": (
        int(seleccion["ventana_dias"])
        == propuesta_guardada["window_days"]
        and seleccion["modelo"]
        == propuesta_guardada["model"]
        and np.isclose(
            seleccion["umbral_f2"],
            propuesta_guardada["threshold"]
        )
    ),
    "Configuración final coincidente": (
        int(seleccion["ventana_dias"])
        == configuracion_final["window_days"]
        and seleccion["modelo"]
        == configuracion_final["model"]
        and np.isclose(
            seleccion["umbral_f2"],
            configuracion_final["threshold"]
        )
    ),
    "Selección de 60 días, L2 y umbral 0,20": (
        int(seleccion["ventana_dias"]) == 60
        and seleccion["modelo"] == "logistic_l2"
        and np.isclose(seleccion["umbral_f2"], 0.20)
    )
}

print("VERIFICACIÓN DE LA SELECCIÓN DEL MODELO")
print("-" * 65)

for nombre, correcto in pruebas_seleccion.items():
    print(f"{nombre}: {'OK' if correcto else 'REVISAR'}")

print("-" * 65)

print(
    "Configuración reproducida:",
    f"{int(seleccion['ventana_dias'])} días,",
    seleccion["modelo"],
    f"umbral {seleccion['umbral_f2']:.2f}"
)

print(
    "Configuraciones equivalentes:",
    len(equivalentes)
)

print(
    f"Comprobaciones superadas: "
    f"{sum(pruebas_seleccion.values())}/"
    f"{len(pruebas_seleccion)}"
)

if all(pruebas_seleccion.values()):
    print("COHERENCIA DE SELECCIÓN: SUPERADA")
else:
    print("COHERENCIA DE SELECCIÓN: REQUIERE REVISIÓN")

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 7

Se han superado las nueve comprobaciones de la regla de selección del modelo definitivo.

La configuración seleccionada corresponde a una regresión logística L2, con 60 días de historial y un umbral de decisión de 0,20. La decisión se reproduce al aplicar los criterios registrados a las nueve configuraciones evaluadas, de las cuales seis cumplen las tolerancias de equivalencia.

Este resultado acredita la coherencia de la selección con el protocolo almacenado, pero no demuestra de forma independiente la fecha en que se establecieron sus criterios.

## 8. Auditoría de trazabilidad de los checkpoints

### Objetivo

Examinar las evidencias disponibles sobre el origen de los nueve checkpoints y determinar hasta qué punto permiten reconstruir las condiciones de su generación.

### Procedimiento

Se consulta el origen registrado para cada configuración, se inspecciona la presencia de metadatos asociados al checkpoint y se calcula una huella digital SHA-256 de los archivos existentes.

### Alcance

Las huellas digitales identifican el contenido de los archivos en el momento de esta comprobación. No constituyen una prueba retrospectiva de los parámetros empleados durante su generación.

La ausencia de un manifiesto completo de entrenamiento impide acreditar únicamente mediante estos archivos la correspondencia exacta con las matrices de entrada, los hiperparámetros y los pliegues de validación originales.

La comprobación es exclusivamente de lectura y no modifica los resultados.


In [ ]:

import hashlib

tabla_origen = pd.read_csv(
    RUTA_PROYECTO
    / "resultados"
    / "modelado_comparacion_temporal_oof.csv"
)

carpeta = (
    RUTA_PROYECTO
    / "resultados"
    / "checkpoints_oof_temporal"
)

huellas_checkpoints = {}
sin_metadatos = 0

print("AUDITORÍA DE TRAZABILIDAD DE CHECKPOINTS")
print("-" * 65)

for _, fila in tabla_origen.iterrows():

    ventana = int(fila["ventana_dias"])
    modelo = str(fila["modelo"])

    nombre = f"oof_{ventana}d_{modelo}.pkl"
    ruta = carpeta / nombre

    if not ruta.is_file():
        print(f"{nombre}: ARCHIVO NO ENCONTRADO")
        continue

    # Calcular la huella digital actual
    calculador = hashlib.sha256()

    with open(ruta, "rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(1024 * 1024),
            b""
        ):
            calculador.update(bloque)

    huella = calculador.hexdigest()
    huellas_checkpoints[nombre] = huella

    # Revisar metadatos del objeto
    checkpoint = pd.read_pickle(ruta)

    atributos = dict(checkpoint.attrs)

    # Buscar archivos auxiliares con nombres asociados
    auxiliares = [
        ruta.with_suffix(".json"),
        ruta.with_suffix(".manifest.json"),
        ruta.with_suffix(".metadata.json")
    ]

    metadatos_auxiliares = [
        archivo.name
        for archivo in auxiliares
        if archivo.is_file()
    ]

    existen_metadatos = bool(
        atributos or metadatos_auxiliares
    )

    if not existen_metadatos:
        sin_metadatos += 1

    print(f"\n{nombre}")
    print(
        "  Origen registrado:",
        fila["origen_probabilidades"]
    )
    print(
        "  SHA-256 (primeros 16 caracteres):",
        huella[:16]
    )
    print(
        "  Metadatos asociados:",
        "ENCONTRADOS" if existen_metadatos
        else "NO ENCONTRADOS"
    )

print("\n" + "-" * 65)

print(
    "Archivos con huella calculada:",
    len(huellas_checkpoints)
)

print(
    "Archivos sin metadatos asociados:",
    sin_metadatos
)

print(
    "La procedencia exacta del entrenamiento "
    "no queda acreditada únicamente con "
    "las probabilidades almacenadas."
)

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 8

Se han identificado los nueve checkpoints temporales y se han calculado sus huellas digitales SHA-256.

Los archivos examinados no incorporan metadatos asociados que permitan acreditar de forma independiente los parámetros, las variables y los pliegues empleados durante su generación.

Esta circunstancia representa una limitación de trazabilidad, pero no demuestra la existencia de errores en las predicciones.

## 9. Reproducción independiente del checkpoint de 60 días y regresión logística L2

### Objetivo

Comprobar si es posible reproducir las predicciones fuera de muestra almacenadas para la configuración definitiva mediante una nueva ejecución de la validación cruzada.

### Procedimiento

Se recuperan las variables correspondientes a 60 días de actividad y los clientes del conjunto de desarrollo.

Se reconstruye la regresión logística L2 con el escalado, los hiperparámetros y los cinco pliegues documentados en el notebook original de modelado.

Las nuevas probabilidades OOF se comparan con las del checkpoint conservado.

### Alcance

La coincidencia de las predicciones proporcionaría evidencia adicional de reproducibilidad de esta configuración concreta.

La prueba no permite demostrar por sí sola cuándo se generó el checkpoint original ni verificar los otros ocho checkpoints.

Los modelos utilizados en esta comprobación son temporales y no se guardan. Los archivos originales permanecen intactos.


In [ ]:

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Recuperar las evidencias originales
datos_60 = pd.read_pickle(
    RUTA_PROYECTO
    / "resultados"
    / "dataset_modelado_60d.pkl"
)

referencia_oof = pd.read_pickle(
    RUTA_PROYECTO
    / "resultados"
    / "modelado_predicciones_oof_temporales.pkl"
)

checkpoint_original = pd.read_pickle(
    RUTA_PROYECTO
    / "resultados"
    / "checkpoints_oof_temporal"
    / "oof_60d_logistic_l2.pkl"
)

variables = pd.read_csv(
    RUTA_PROYECTO
    / "resultados"
    / "columnas_predictoras_176_90d.csv"
)["variable"].tolist()

# Alinear clientes con el conjunto de desarrollo
datos_alineados = referencia_oof[
    ["customer_id", "churn_real"]
].merge(
    datos_60[
        ["customer_id", "churn", *variables]
    ],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

if len(datos_alineados) != 44_203:
    raise ValueError("Tamaño de desarrollo inesperado.")

if not np.array_equal(
    datos_alineados["churn_real"].to_numpy(),
    datos_alineados["churn"].to_numpy()
):
    raise ValueError("Las etiquetas churn no coinciden.")

if not np.array_equal(
    referencia_oof["customer_id"].to_numpy(),
    checkpoint_original["customer_id"].to_numpy()
):
    raise ValueError("El orden de los clientes no coincide.")

X = datos_alineados[variables]
y = datos_alineados["churn_real"].to_numpy(dtype=int)

if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Existen variables no finitas.")

# Reconstruir el modelo original
modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="liblinear",
        max_iter=200,
        random_state=42
    ))
])

# Reconstruir los cinco pliegues
cv_reproducida = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=314159
)

# Generar nuevamente las probabilidades OOF
probabilidades_nuevas = cross_val_predict(
    estimator=modelo,
    X=X,
    y=y,
    cv=cv_reproducida,
    method="predict_proba",
    n_jobs=1
)[:, 1]

probabilidades_originales = (
    checkpoint_original["probabilidad"]
    .to_numpy(dtype=float)
)

# Comparar predicciones
diferencias = np.abs(
    probabilidades_nuevas
    - probabilidades_originales
)

coincidencia = np.allclose(
    probabilidades_nuevas,
    probabilidades_originales,
    rtol=1e-8,
    atol=1e-8
)

clases_nuevas = (
    probabilidades_nuevas >= 0.20
)

clases_originales = (
    probabilidades_originales >= 0.20
)

diferencias_clasificacion = int(
    np.sum(clases_nuevas != clases_originales)
)

print("REPRODUCCIÓN DEL CHECKPOINT 60D LOGISTIC_L2")
print("-" * 65)

print("Clientes evaluados:", len(y))
print("Variables empleadas:", X.shape[1])
print("Pliegues de validación:", 5)

print(
    "Máxima diferencia absoluta:",
    f"{diferencias.max():.12g}"
)

print(
    "Diferencia absoluta media:",
    f"{diferencias.mean():.12g}"
)

print(
    "Clasificaciones distintas con umbral 0,20:",
    diferencias_clasificacion
)

print("-" * 65)

print(
    "Probabilidades reproducidas:",
    "OK" if coincidencia else "REVISAR"
)

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 9

La reproducción independiente del checkpoint correspondiente a la regresión logística L2 con 60 días de historial ha resultado satisfactoria.

Las 44.203 probabilidades obtenidas mediante una nueva validación cruzada de cinco pliegues coinciden exactamente con las almacenadas en el checkpoint original. No se han detectado diferencias numéricas ni discrepancias en las clasificaciones realizadas con el umbral de 0,20.

Este resultado aporta evidencia directa de reproducibilidad para la configuración definitiva. No acredita por sí solo la fecha de ejecución original ni la reproducibilidad de las restantes configuraciones.

## 10. Reproducción independiente de la regresión logística ponderada de 60 días

### Objetivo

Comprobar si las predicciones OOF almacenadas para la regresión logística con ponderación de clases pueden reproducirse utilizando los datos y la configuración documentados en el análisis original.

### Procedimiento

Se utiliza el mismo conjunto de desarrollo de 44.203 clientes y 176 variables, manteniendo cinco pliegues estratificados, mezcla aleatoria y semilla 314159.

Se reconstruye un modelo de regresión logística L2 con escalado de variables y ponderación automática de clases mediante class_weight="balanced".

Las nuevas probabilidades se comparan con las almacenadas en el checkpoint original de 60 días.

### Alcance

La coincidencia de las probabilidades aportaría evidencia adicional de reproducibilidad del procedimiento de modelado.

La comprobación no modifica los archivos originales ni sustituye las predicciones conservadas.


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

# Recuperar el checkpoint original
ruta_balanceado_60 = (
    RUTA_PROYECTO
    / "resultados"
    / "checkpoints_oof_temporal"
    / "oof_60d_logistic_balanced.pkl"
)

checkpoint_balanceado_60 = pd.read_pickle(
    ruta_balanceado_60
)

# Verificar la correspondencia con el desarrollo
if len(checkpoint_balanceado_60) != len(y):
    raise ValueError(
        "El número de clientes no coincide."
    )

if not np.array_equal(
    checkpoint_balanceado_60["customer_id"]
    .astype("string").to_numpy(),
    referencia_oof["customer_id"]
    .astype("string").to_numpy()
):
    raise ValueError(
        "Los identificadores no coinciden."
    )

if not np.array_equal(
    checkpoint_balanceado_60["churn_real"].to_numpy(),
    y
):
    raise ValueError(
        "Las etiquetas churn no coinciden."
    )

# Configuración documentada del modelo original
modelo_balanceado_60 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=200,
        random_state=42
    ))
])

# Repetir la validación cruzada original
probabilidades_balanceadas_nuevas = cross_val_predict(
    estimator=modelo_balanceado_60,
    X=X,
    y=y,
    cv=cv_reproducida,
    method="predict_proba",
    n_jobs=1
)[:, 1]

# Recuperar las probabilidades originales
probabilidades_balanceadas_originales = (
    checkpoint_balanceado_60["probabilidad"]
    .to_numpy(dtype=float)
)

# Comparar resultados
diferencias_balanceado = np.abs(
    probabilidades_balanceadas_nuevas
    - probabilidades_balanceadas_originales
)

coincidencia_exacta = np.array_equal(
    probabilidades_balanceadas_nuevas,
    probabilidades_balanceadas_originales
)

coincidencia_numerica = np.allclose(
    probabilidades_balanceadas_nuevas,
    probabilidades_balanceadas_originales,
    rtol=1e-8,
    atol=1e-8
)

print("REPRODUCCIÓN CHECKPOINT 60D LOGISTIC_BALANCED")
print("-" * 65)

print("Clientes evaluados:", len(y))
print("Variables utilizadas:", X.shape[1])
print("Pliegues de validación:", 5)

print(
    "Diferencia máxima absoluta:",
    f"{diferencias_balanceado.max():.12g}"
)

print(
    "Diferencia media absoluta:",
    f"{diferencias_balanceado.mean():.12g}"
)

print("-" * 65)

print(
    "Coincidencia exacta:",
    "SÍ" if coincidencia_exacta else "NO"
)

print(
    "Coincidencia numérica:",
    "OK" if coincidencia_numerica else "REVISAR"
)

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 10

La regresión logística con ponderación de clases y 60 días de historial se ha reproducido exactamente.

Las 44.203 probabilidades generadas mediante una nueva validación cruzada coinciden con las almacenadas en el checkpoint original, sin diferencias numéricas.

Con esta comprobación, los dos modelos de regresión logística de 60 días cuentan con evidencia directa de reproducibilidad.

## 11. Reproducción independiente de XGBoost con 60 días

### Objetivo

Verificar si las predicciones OOF almacenadas para XGBoost con 60 días de historial pueden reproducirse mediante una nueva validación cruzada.

### Procedimiento

Se utilizan los mismos 44.203 clientes, 176 variables predictoras y cinco pliegues estratificados de las comprobaciones anteriores.

Se reconstruye XGBoost con los hiperparámetros documentados en el análisis original y se comparan las nuevas probabilidades con las conservadas en el checkpoint.

### Alcance

Esta comprobación permite evaluar la reproducibilidad de la tercera configuración finalista de 60 días.

Una discrepancia numérica deberá investigarse considerando también la versión de XGBoost y las condiciones de ejecución.

El procedimiento no modifica los modelos ni los resultados originales.


In [ ]:

from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_predict

# Recuperar el checkpoint original
ruta_xgboost_60 = (
    RUTA_PROYECTO
    / "resultados"
    / "checkpoints_oof_temporal"
    / "oof_60d_xgboost.pkl"
)

checkpoint_xgboost_60 = pd.read_pickle(
    ruta_xgboost_60
)

# Comprobar la alineación
if not np.array_equal(
    checkpoint_xgboost_60["customer_id"]
    .astype("string").to_numpy(),
    referencia_oof["customer_id"]
    .astype("string").to_numpy()
):
    raise ValueError(
        "Los identificadores de clientes no coinciden."
    )

if not np.array_equal(
    checkpoint_xgboost_60["churn_real"].to_numpy(),
    y
):
    raise ValueError(
        "Las etiquetas churn no coinciden."
    )

# Configuración documentada en el notebook original
modelo_xgboost_60 = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

# Reproducir las predicciones OOF
probabilidades_xgboost_nuevas = cross_val_predict(
    estimator=modelo_xgboost_60,
    X=X,
    y=y,
    cv=cv_reproducida,
    method="predict_proba",
    n_jobs=1
)[:, 1]

probabilidades_xgboost_originales = (
    checkpoint_xgboost_60["probabilidad"]
    .to_numpy(dtype=float)
)

# Comparar probabilidades
diferencias_xgboost = np.abs(
    probabilidades_xgboost_nuevas
    - probabilidades_xgboost_originales
)

exacta_xgboost = np.array_equal(
    probabilidades_xgboost_nuevas,
    probabilidades_xgboost_originales
)

numerica_xgboost = np.allclose(
    probabilidades_xgboost_nuevas,
    probabilidades_xgboost_originales,
    rtol=1e-8,
    atol=1e-8
)

print("REPRODUCCIÓN CHECKPOINT 60D XGBOOST")
print("-" * 65)

print("Clientes evaluados:", len(y))
print("Variables utilizadas:", X.shape[1])
print("Pliegues de validación:", 5)

print(
    "Diferencia máxima absoluta:",
    f"{diferencias_xgboost.max():.12g}"
)

print(
    "Diferencia media absoluta:",
    f"{diferencias_xgboost.mean():.12g}"
)

print("-" * 65)

print(
    "Coincidencia exacta:",
    "SÍ" if exacta_xgboost else "NO"
)

print(
    "Coincidencia numérica:",
    "OK" if numerica_xgboost else "REVISAR"
)

print("No se ha modificado ningún archivo.")



### Interpretación de la comprobación 11

La reproducción independiente del modelo XGBoost con 60 días de historial ha resultado satisfactoria.

Las 44.203 probabilidades generadas mediante cinco pliegues de validación cruzada coinciden exactamente con las almacenadas en el checkpoint original. No se han detectado diferencias numéricas.

Con esta comprobación se completa la reproducción independiente de los tres modelos evaluados con 60 días de actividad.

## 12. Conclusiones de la auditoría técnica independiente

Las once comprobaciones realizadas han aportado evidencias favorables sobre la consistencia interna y la reproducibilidad del análisis desarrollado.

Se ha verificado la disponibilidad de los archivos principales, la correspondencia de los nueve checkpoints con las predicciones consolidadas, la reproducción de las métricas temporales y la coherencia de las particiones de datos.

Asimismo, se ha reconstruido el esquema de validación cruzada estratificada y se ha confirmado que el modelo final almacenado reproduce las predicciones y las principales métricas del conjunto de prueba externa.

La aplicación de los criterios de selección documentados conduce nuevamente a la regresión logística L2 con 60 días de historial y umbral de decisión de 0,20.

Finalmente, las predicciones OOF de las tres configuraciones de 60 días se han reproducido exactamente mediante nuevas ejecuciones de validación cruzada.

### Limitaciones de la verificación

Los resultados obtenidos no permiten descartar todas las posibles fugas de información ni acreditar retrospectivamente la fecha en que se establecieron los criterios de selección.

Tampoco resuelven la falta de documentación suficiente sobre la definición de la etiqueta churn y la dirección cronológica de los registros diarios.

Aunque se ha comprobado la consistencia de los nueve checkpoints, la reproducción independiente del entrenamiento se ha realizado únicamente para las tres configuraciones de 60 días.

### Conclusión general

Las comprobaciones realizadas proporcionan evidencias sólidas de consistencia numérica y reproducibilidad del modelo definitivo y de las configuraciones de 60 días.

Estas evidencias refuerzan la fiabilidad técnica del estudio, sin eliminar las limitaciones metodológicas identificadas.

La auditoría se ha efectuado de forma independiente, conservando los archivos, modelos y resultados originales.
